# 🔍 Schema Validation Agent — Multi-Agent Data Quality System

Questo notebook implementa il primo agente del sistema multi-agente per la **Data Quality** su dataset NoiPA.

## Architettura
Lo **Schema Validation Agent** è costruito con:
- **LangGraph** per orchestrare il flusso dell'agente come grafo di stati
- **Google Gemini 2.0 Flash** (via `langchain-google-genai`) per l'analisi intelligente dello schema
- **pandas** per la manipolazione del DataFrame

## Cosa fa questo agente
1. Riceve un DataFrame pandas in input
2. Usa Gemini per analizzare lo schema e inferire i tipi attesi
3. Verifica le naming convention delle colonne (snake_case, no spazi, no caratteri speciali)
4. Confronta i tipi attesi con i tipi effettivi trovati nei dati
5. Restituisce un report strutturato per colonna con violazioni e suggerimenti

## 📦 Installazione dipendenze

Installa tutti i pacchetti necessari. Esegui questa cella solo la prima volta.

In [13]:
!pip install -q \
    langchain-google-genai \
    langgraph \
    langchain-core \
    python-dotenv \
    pandas


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 🔑 Configurazione API Key

La chiave API di Google Gemini viene caricata da un file `.env` nella root del progetto.

Crea un file `.env` con questo contenuto:
```
GOOGLE_API_KEY=la_tua_api_key_qui
```

> Puoi ottenere una chiave gratuita su [Google AI Studio](https://aistudio.google.com/app/apikey).

In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError(
        "❌ GOOGLE_API_KEY non trovata. "
        "Assicurati di avere un file .env con GOOGLE_API_KEY=..."
    )

print("✅ API Key caricata correttamente.")

✅ API Key caricata correttamente.


## 📥 Caricamento del Dataset

Carichiamo il CSV di test NoiPA da `data/dataset_noipa_1.csv`. La cartella `data/` deve essere nella stessa directory del notebook.

In [15]:
import pandas as pd

DATASET_PATH = "data/dataset_noipa_1.csv"

df = pd.read_csv(DATASET_PATH, dtype=str)  # dtype=str per preservare i valori "sporchi"

print(f"✅ Dataset caricato: {df.shape[0]} righe × {df.shape[1]} colonne")
print(f"📋 Colonne: {list(df.columns)}")
df.head()

✅ Dataset caricato: 51 righe × 6 colonne
📋 Colonne: ['Rata', 'Ente', 'Descrizione', 'Tipo_imposta', 'Imposta', 'Spesa']


,Rata,Ente,Descrizione,Tipo_imposta,Imposta,Spesa
0,202501,101,Ministero dell'Economia e delle Finanze,Erariali,IRPEF,125430.50
1,202501,102,Ministero della Salute,Previdenziali,INPS,98234.75
2,202502,103,Ministero dell'Istruzione,Erariali,IRAP,43210.00
3,202502,104,Ministero della Difesa,Varie,Addizionali Comunali,12345.60
4,202503,105,Ministero della Giustizia,Previdenziali,INAIL,8765.30


## 🧱 Definizione dello Stato LangGraph (`TypedDict`)

In LangGraph, lo **stato** è un `TypedDict` che fluisce tra i nodi del grafo.

Il nostro stato contiene:
- `dataframe`: il DataFrame pandas originale
- `schema_summary`: un riassunto testuale dello schema passato a Gemini
- `gemini_analysis`: la risposta grezza di Gemini
- `naming_violations`: lista di colonne con naming convention errate
- `validation_results`: il dizionario finale con i risultati per colonna
- `errors`: lista di eventuali errori di esecuzione

In [16]:
from typing import TypedDict, Optional
import pandas as pd


class SchemaValidationState(TypedDict):
    """Stato condiviso tra tutti i nodi del grafo Schema Validation Agent."""
    dataframe: pd.DataFrame                    # DataFrame originale in input
    schema_summary: str                        # Riassunto testuale dello schema
    gemini_analysis: str                       # Risposta raw di Gemini
    naming_violations: list[dict]              # Colonne con naming convention errate
    validation_results: dict[str, dict]        # Risultati finali per colonna
    errors: list[str]                          # Errori di esecuzione

## 🔧 Funzioni di supporto

Definiamo le utility che verranno usate dai nodi del grafo:

- `build_schema_summary()` — costruisce una rappresentazione testuale dello schema da passare a Gemini
- `check_naming_conventions()` — verifica snake_case, spazi e caratteri speciali
- `parse_gemini_response()` — estrae i tipi attesi dalla risposta JSON di Gemini

In [17]:
import re
import json


def build_schema_summary(df: pd.DataFrame) -> str:
    """
    Costruisce una stringa descrittiva dello schema del DataFrame,
    includendo nome colonna, tipo pandas e campione di valori.
    Serve come contesto per Gemini.
    """
    lines = ["Dataset schema (colonne, tipo pandas, campione di 5 valori non-null):"]
    for col in df.columns:
        sample_values = df[col].dropna().head(5).tolist()
        lines.append(
            f"  - '{col}': tipo={df[col].dtype}, campione={sample_values}"
        )
    return "\n".join(lines)


def check_naming_conventions(columns: list[str]) -> list[dict]:
    """
    Controlla che ogni nome colonna rispetti le naming convention:
      1. Nessuno spazio
      2. Solo caratteri alfanumerici e underscore (no -, ., etc.)
      3. Formato snake_case (tutto minuscolo con underscore)

    Restituisce una lista di dizionari con le violazioni trovate.
    """
    violations = []
    pattern_valid = re.compile(r'^[a-zA-Z][a-zA-Z0-9_]*$')  # no spazi, no char speciali
    pattern_snake = re.compile(r'^[a-z][a-z0-9_]*$')         # snake_case puro

    for col in columns:
        col_violations = []
        suggestions = []

        if ' ' in col:
            col_violations.append("Contiene spazi")
            suggestions.append(f"Rinomina in '{col.replace(' ', '_').lower()}'")

        if not pattern_valid.match(col):
            col_violations.append("Contiene caratteri speciali non permessi")
            clean = re.sub(r'[^a-zA-Z0-9_]', '_', col).lower()
            suggestions.append(f"Rinomina in '{clean}'")

        if pattern_valid.match(col) and not pattern_snake.match(col):
            col_violations.append("Non è in snake_case (lettere maiuscole presenti)")
            suggestions.append(f"Rinomina in '{col.lower()}'")

        if col_violations:
            violations.append({
                "colonna": col,
                "violazioni": col_violations,
                "suggerimenti": list(set(suggestions))
            })

    return violations


def parse_gemini_response(raw_response: str) -> dict:
    """
    Estrae il JSON dalla risposta di Gemini.
    Gemini può includere testo prima/dopo il JSON — lo isoliamo
    cercando il primo '{' e l'ultimo '}'.
    Restituisce un dict vuoto in caso di fallimento.
    """
    try:
        start = raw_response.index('{')
        end = raw_response.rindex('}') + 1
        json_str = raw_response[start:end]
        return json.loads(json_str)
    except (ValueError, json.JSONDecodeError) as e:
        print(f"⚠️  Impossibile parsare la risposta JSON di Gemini: {e}")
        return {}


print("✅ Funzioni di supporto definite.")

✅ Funzioni di supporto definite.


## 🤖 Inizializzazione del modello Gemini

Usiamo `ChatGoogleGenerativeAI` da `langchain-google-genai` con il modello `gemini-2.0-flash`.

> **Perché Flash?** È il modello più veloce ed economico di Gemini 2.0, ideale per task di analisi strutturata come questo.

In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0,          # Temperatura 0 per risposte deterministiche e strutturate
    max_output_tokens=4096,
)

print("✅ Modello Gemini 2.0 Flash inizializzato.")

✅ Modello Gemini 2.0 Flash inizializzato.


## 🧩 Nodi del Grafo LangGraph

Il grafo è composto da **4 nodi**, ognuno dei quali legge e scrive sullo stato condiviso `SchemaValidationState`:

| Nodo | Responsabilità |
|------|----------------|
| `node_build_schema` | Costruisce il riassunto testuale dello schema |
| `node_gemini_analysis` | Chiama Gemini per inferire i tipi attesi |
| `node_naming_check` | Verifica le naming convention delle colonne |
| `node_compile_results` | Combina tutto nel dizionario finale dei risultati |

In [19]:
from langchain_core.messages import HumanMessage


# ─────────────────────────────────────────────────
# NODO 1: Costruisce il riassunto dello schema
# ─────────────────────────────────────────────────
def node_build_schema(state: SchemaValidationState) -> SchemaValidationState:
    """
    Nodo 1: Legge il DataFrame e produce una stringa descrittiva
    dello schema che verrà passata a Gemini come contesto.
    """
    print("[Nodo 1] 📐 Costruzione schema summary...")
    summary = build_schema_summary(state["dataframe"])
    return {**state, "schema_summary": summary}


# ─────────────────────────────────────────────────
# NODO 2: Analisi Gemini — inferisce i tipi attesi
# ─────────────────────────────────────────────────
def node_gemini_analysis(state: SchemaValidationState) -> SchemaValidationState:
    """
    Nodo 2: Invia lo schema a Gemini con un prompt strutturato.
    Chiede di restituire un JSON con, per ogni colonna:
      - tipo_atteso: il tipo di dato che ci si aspetta (string, integer, float, date)
      - formato_atteso: eventuale formato specifico (es. YYYYMM)
      - valori_ammessi: lista di valori validi se la colonna è categorica
      - note: commento semantico sul ruolo della colonna
    """
    print("[Nodo 2] 🤖 Analisi Gemini in corso...")

    prompt = f"""
Sei un esperto di qualità dei dati per sistemi della Pubblica Amministrazione italiana.
Ti viene fornito lo schema di un dataset NoiPA (piattaforma MEF per gli stipendi PA).

{state['schema_summary']}

Il tuo compito è analizzare ogni colonna e restituire SOLO un oggetto JSON valido
(senza testo aggiuntivo prima o dopo, senza markdown, senza backtick).

Per ogni colonna restituisci un oggetto con questa struttura:
{{
  "<nome_colonna>": {{
    "tipo_atteso": "<string|integer|float|date|categorical>",
    "formato_atteso": "<descrizione del formato atteso, es. YYYYMM, oppure null>",
    "valori_ammessi": ["<lista di valori validi se categorica, altrimenti []>"],
    "note": "<breve descrizione semantica del ruolo della colonna>"
  }}
}}

Colonne da analizzare: {list(state['dataframe'].columns)}

Rispondi SOLO con il JSON, niente altro.
"""

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        gemini_raw = response.content
        print("[Nodo 2] ✅ Risposta Gemini ricevuta.")
    except Exception as e:
        error_msg = f"Errore chiamata Gemini: {str(e)}"
        print(f"[Nodo 2] ❌ {error_msg}")
        gemini_raw = "{}"
        state["errors"].append(error_msg)

    return {**state, "gemini_analysis": gemini_raw}


# ─────────────────────────────────────────────────
# NODO 3: Controllo naming convention
# ─────────────────────────────────────────────────
def node_naming_check(state: SchemaValidationState) -> SchemaValidationState:
    """
    Nodo 3: Verifica che i nomi delle colonne rispettino le convenzioni:
    - Nessuno spazio
    - Solo caratteri alfanumerici e underscore
    - snake_case (tutto minuscolo)
    """
    print("[Nodo 3] 🏷️  Controllo naming convention...")
    violations = check_naming_conventions(list(state["dataframe"].columns))
    if violations:
        print(f"[Nodo 3] ⚠️  Trovate {len(violations)} violazioni naming.")
    else:
        print("[Nodo 3] ✅ Tutte le colonne rispettano la naming convention.")
    return {**state, "naming_violations": violations}


# ─────────────────────────────────────────────────
# NODO 4: Compilazione risultati finali
# ─────────────────────────────────────────────────
def node_compile_results(state: SchemaValidationState) -> SchemaValidationState:
    """
    Nodo 4: Combina i risultati di Gemini e del naming check.
    Per ogni colonna produce un dizionario con:
      - tipo_atteso: da Gemini
      - tipo_effettivo: rilevato da pandas
      - formato_atteso: da Gemini
      - valori_ammessi: da Gemini
      - note_semantiche: da Gemini
      - violazioni_naming: dal naming check
      - violazioni_tipo: confronto tipo atteso vs effettivo
      - violazioni_valori: valori non conformi trovati
      - suggerimenti: lista consolidata
    """
    print("[Nodo 4] 📊 Compilazione risultati finali...")

    df = state["dataframe"]
    gemini_data = parse_gemini_response(state["gemini_analysis"])
    naming_violations_map = {
        v["colonna"]: v for v in state["naming_violations"]
    }

    # Mappa da tipo_atteso Gemini → tipi pandas compatibili
    tipo_to_pandas = {
        "integer":     ["int64", "Int64", "int32", "int16"],
        "float":       ["float64", "float32"],
        "string":      ["object"],
        "date":        ["datetime64[ns]", "object"],
        "categorical": ["object", "category"],
    }

    results = {}

    for col in df.columns:
        gem = gemini_data.get(col, {})
        tipo_atteso     = gem.get("tipo_atteso", "sconosciuto")
        formato_atteso  = gem.get("formato_atteso", None)
        valori_ammessi  = gem.get("valori_ammessi", [])
        note_semantiche = gem.get("note", "")
        tipo_effettivo  = str(df[col].dtype)

        violazioni_tipo = []
        violazioni_valori = []
        suggerimenti = []

        # ── Verifica tipo atteso vs effettivo ──────────────────────────
        pandas_compatibili = tipo_to_pandas.get(tipo_atteso, [])
        if pandas_compatibili and tipo_effettivo not in pandas_compatibili:

            # Prova a vedere se i valori sono convertibili al tipo atteso
            if tipo_atteso in ("integer", "float"):
                non_numerici = df[col].dropna()
                non_numerici = non_numerici[pd.to_numeric(non_numerici, errors='coerce').isna()]
                if len(non_numerici) > 0:
                    violazioni_tipo.append(
                        f"Tipo atteso '{tipo_atteso}' ma trovato '{tipo_effettivo}'. "
                        f"{len(non_numerici)} valori non convertibili a numerico."
                    )
                    esempi = non_numerici.head(3).tolist()
                    violazioni_valori.extend([str(v) for v in esempi])
                    suggerimenti.append(
                        f"Correggere i valori non numerici in '{col}': {esempi}"
                    )

        # ── Verifica formato per colonne date (es. YYYYMM) ────────────
        if formato_atteso and "YYYYMM" in str(formato_atteso).upper():
            pattern_yyyymm = re.compile(r'^\d{6}$')
            non_conformi = df[col].dropna()
            non_conformi = non_conformi[~non_conformi.astype(str).str.match(pattern_yyyymm)]
            if len(non_conformi) > 0:
                violazioni_tipo.append(
                    f"Formato atteso YYYYMM ma trovati {len(non_conformi)} valori non conformi."
                )
                esempi = non_conformi.head(3).tolist()
                violazioni_valori.extend([str(v) for v in esempi])
                suggerimenti.append(
                    f"Normalizzare il formato di '{col}' in YYYYMM (es. 202501). "
                    f"Valori problematici: {esempi}"
                )

        # ── Verifica valori categorici ammessi ────────────────────────
        if valori_ammessi:
            valori_non_validi = df[col].dropna()
            valori_non_validi = valori_non_validi[
                ~valori_non_validi.astype(str).isin([str(v) for v in valori_ammessi])
            ]
            if len(valori_non_validi) > 0:
                unici = valori_non_validi.unique().tolist()[:5]
                violazioni_valori.extend([str(v) for v in unici])
                suggerimenti.append(
                    f"'{col}' contiene valori fuori dal dominio atteso {valori_ammessi}. "
                    f"Trovati: {unici}"
                )

        # ── Aggiungi violazioni naming se presenti ────────────────────
        naming_info = naming_violations_map.get(col, {})
        if naming_info:
            suggerimenti.extend(naming_info.get("suggerimenti", []))

        # ── Verifica valori nulli ─────────────────────────────────────
        null_count = df[col].isna().sum()
        null_pct = round(null_count / len(df) * 100, 1)
        if null_count > 0:
            suggerimenti.append(
                f"'{col}' ha {null_count} valori nulli ({null_pct}%). Valutare imputation o rimozione."
            )

        results[col] = {
            "tipo_atteso":          tipo_atteso,
            "tipo_effettivo":       tipo_effettivo,
            "formato_atteso":       formato_atteso,
            "valori_ammessi":       valori_ammessi,
            "note_semantiche":      note_semantiche,
            "null_count":           int(null_count),
            "null_pct":             null_pct,
            "violazioni_naming":    naming_info.get("violazioni", []),
            "violazioni_tipo":      violazioni_tipo,
            "violazioni_valori":    list(set(violazioni_valori)),
            "suggerimenti":         list(set(suggerimenti)),
            "status":               "✅ OK" if not (violazioni_tipo or violazioni_valori or naming_info) else "⚠️ ANOMALIE"
        }

    print(f"[Nodo 4] ✅ Risultati compilati per {len(results)} colonne.")
    return {**state, "validation_results": results}


print("✅ Nodi del grafo definiti.")

✅ Nodi del grafo definiti.


## 🔗 Costruzione del Grafo LangGraph

Colleghiamo i 4 nodi in un grafo sequenziale:

```
START → build_schema → gemini_analysis → naming_check → compile_results → END
```

Ogni nodo riceve lo stato aggiornato dal nodo precedente e lo arricchisce con nuove informazioni.

In [20]:
from langgraph.graph import StateGraph, END

# Inizializza il builder del grafo con il nostro TypedDict come schema di stato
builder = StateGraph(SchemaValidationState)

# Registra i nodi
builder.add_node("build_schema",     node_build_schema)
builder.add_node("gemini_analysis",  node_gemini_analysis)
builder.add_node("naming_check",     node_naming_check)
builder.add_node("compile_results",  node_compile_results)

# Definisce il flusso sequenziale
builder.set_entry_point("build_schema")
builder.add_edge("build_schema",    "gemini_analysis")
builder.add_edge("gemini_analysis", "naming_check")
builder.add_edge("naming_check",    "compile_results")
builder.add_edge("compile_results", END)

# Compila il grafo in un eseguibile
schema_validation_graph = builder.compile()

print("✅ Grafo LangGraph compilato.")

✅ Grafo LangGraph compilato.


## 🚀 Esecuzione dell'Agente

Prepariamo lo stato iniziale e avviamo il grafo. Lo stato iniziale contiene solo il DataFrame — tutti gli altri campi verranno popolati dai nodi durante l'esecuzione.

In [21]:
# Stato iniziale: solo il DataFrame, tutto il resto viene popolato dai nodi
initial_state: SchemaValidationState = {
    "dataframe":          df,
    "schema_summary":     "",
    "gemini_analysis":    "",
    "naming_violations":  [],
    "validation_results": {},
    "errors":             [],
}

print("🚀 Avvio Schema Validation Agent...\n")

# Esegui il grafo
final_state = schema_validation_graph.invoke(initial_state)

print("\n✅ Agente completato.")
if final_state["errors"]:
    print(f"⚠️  Errori riscontrati: {final_state['errors']}")

🚀 Avvio Schema Validation Agent...

[Nodo 1] 📐 Costruzione schema summary...
[Nodo 2] 🤖 Analisi Gemini in corso...
[Nodo 2] ❌ Errore chiamata Gemini: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 21.119483288s.', 'status': 'RESOURCE_EXHAUSTED', 'details

## 📋 Visualizzazione Risultati

Mostriamo i risultati finali in modo leggibile: prima un riepilogo tabellare, poi il dettaglio per le colonne con anomalie.

In [22]:
import json

results = final_state["validation_results"]

# ── Riepilogo tabellare ────────────────────────────────────────────
print("=" * 70)
print("📊 SCHEMA VALIDATION REPORT — RIEPILOGO")
print("=" * 70)
print(f"{'Colonna':<30} {'Tipo Atteso':<15} {'Tipo Eff.':<12} {'Nulli':>6}  {'Status'}")
print("-" * 70)

for col, info in results.items():
    print(
        f"{col:<30} "
        f"{info['tipo_atteso']:<15} "
        f"{info['tipo_effettivo']:<12} "
        f"{info['null_count']:>4} ({info['null_pct']:>4.1f}%)  "
        f"{info['status']}"
    )

print("=" * 70)

# ── Dettaglio anomalie ─────────────────────────────────────────────
print("\n📋 DETTAGLIO ANOMALIE PER COLONNA:\n")
anomalie_trovate = False

for col, info in results.items():
    ha_anomalie = (
        info["violazioni_naming"] or
        info["violazioni_tipo"] or
        info["violazioni_valori"] or
        info["null_count"] > 0
    )
    if ha_anomalie:
        anomalie_trovate = True
        print(f"  🔴 Colonna: '{col}'")
        if info["note_semantiche"]:
            print(f"     📌 Ruolo: {info['note_semantiche']}")
        if info["violazioni_naming"]:
            print(f"     🏷️  Naming: {info['violazioni_naming']}")
        if info["violazioni_tipo"]:
            print(f"     📐 Tipo:   {info['violazioni_tipo']}")
        if info["violazioni_valori"]:
            print(f"     ❌ Valori non conformi (campione): {info['violazioni_valori']}")
        if info["null_count"] > 0:
            print(f"     🔲 Nulli:  {info['null_count']} ({info['null_pct']}%)")
        if info["suggerimenti"]:
            print("     💡 Suggerimenti:")
            for s in info["suggerimenti"]:
                print(f"        → {s}")
        print()

if not anomalie_trovate:
    print("  ✅ Nessuna anomalia rilevata in nessuna colonna.")

📊 SCHEMA VALIDATION REPORT — RIEPILOGO
Colonna                        Tipo Atteso     Tipo Eff.     Nulli  Status
----------------------------------------------------------------------
Rata                           sconosciuto     str             0 ( 0.0%)  ⚠️ ANOMALIE
Ente                           sconosciuto     str             1 ( 2.0%)  ⚠️ ANOMALIE
Descrizione                    sconosciuto     str             1 ( 2.0%)  ⚠️ ANOMALIE
Tipo_imposta                   sconosciuto     str             1 ( 2.0%)  ⚠️ ANOMALIE
Imposta                        sconosciuto     str             0 ( 0.0%)  ⚠️ ANOMALIE
Spesa                          sconosciuto     str             0 ( 0.0%)  ⚠️ ANOMALIE

📋 DETTAGLIO ANOMALIE PER COLONNA:

  🔴 Colonna: 'Rata'
     🏷️  Naming: ['Non è in snake_case (lettere maiuscole presenti)']
     💡 Suggerimenti:
        → Rinomina in 'rata'

  🔴 Colonna: 'Ente'
     🏷️  Naming: ['Non è in snake_case (lettere maiuscole presenti)']
     🔲 Nulli:  1 (2.0%)
     💡 S

## 💾 Esportazione risultati in JSON

Salviamo il report completo su file JSON per poterlo passare al prossimo agente nel multi-agent pipeline.

In [23]:
import json
from datetime import datetime

output = {
    "agent": "SchemaValidationAgent",
    "dataset": DATASET_PATH,
    "timestamp": datetime.now().isoformat(),
    "righe_totali": len(df),
    "colonne_totali": len(df.columns),
    "colonne_con_anomalie": sum(
        1 for v in results.values() if v["status"] != "✅ OK"
    ),
    "errori_esecuzione": final_state["errors"],
    "risultati_per_colonna": results,
}

output_path = "data/schema_validation_report.json"
os.makedirs("data", exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"✅ Report salvato in: {output_path}")
print(f"   → {output['colonne_totali']} colonne analizzate")
print(f"   → {output['colonne_con_anomalie']} colonne con anomalie")

✅ Report salvato in: data/schema_validation_report.json
   → 6 colonne analizzate
   → 6 colonne con anomalie


## 🔍 Ispezione del dizionario risultati raw

Cella opzionale per ispezionare il dizionario grezzo restituito dall'agente, utile per il debug o per passare i dati al prossimo agente.

In [24]:
# Stampa il JSON completo dei risultati (utile per debug e integrazione con altri agenti)
print(json.dumps(results, ensure_ascii=False, indent=2))

{
  "Rata": {
    "tipo_atteso": "sconosciuto",
    "tipo_effettivo": "str",
    "formato_atteso": null,
    "valori_ammessi": [],
    "note_semantiche": "",
    "null_count": 0,
    "null_pct": 0.0,
    "violazioni_naming": [
      "Non è in snake_case (lettere maiuscole presenti)"
    ],
    "violazioni_tipo": [],
    "violazioni_valori": [],
    "suggerimenti": [
      "Rinomina in 'rata'"
    ],
    "status": "⚠️ ANOMALIE"
  },
  "Ente": {
    "tipo_atteso": "sconosciuto",
    "tipo_effettivo": "str",
    "formato_atteso": null,
    "valori_ammessi": [],
    "note_semantiche": "",
    "null_count": 1,
    "null_pct": 2.0,
    "violazioni_naming": [
      "Non è in snake_case (lettere maiuscole presenti)"
    ],
    "violazioni_tipo": [],
    "violazioni_valori": [],
    "suggerimenti": [
      "Rinomina in 'ente'",
      "'Ente' ha 1 valori nulli (2.0%). Valutare imputation o rimozione."
    ],
    "status": "⚠️ ANOMALIE"
  },
  "Descrizione": {
    "tipo_atteso": "sconosciuto",
  